# Exercise 6: An LLM as a recogniser

*Sequence Labelling · version 0.1.0*

> Annotate 50 Kyrgyz (or Tatar) news sentences with `PER`, `LOC` and `ORG` by hand. Run TurkicNLP with `ner_backend="llm"` and two models of different sizes, and compute entity-level F1 with `seqeval`. Then run the same LLM backend on 50 sentences of the KazNERD test set, where you can compare it with the trained Stanza model. Declare a mapping from KazNERD types to PER/LOC/ORG before scoring, use the same mention boundaries, and apply that mapping to both systems and gold. How large is the gap, and which entity types does the LLM miss?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" "stanza==1.11.0" sklearn-crfsuite transformers sentencepiece requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`, which the first cell uses instead). Stanza is pinned to 1.11.0 because the worked solutions use its Kazakh KazNERD checkpoint; models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`), and missing ones are downloaded. The datasets are read from commit-pinned GitHub files (KazNERD, KyrgyzNER, the Starlang Turkish NER data, TRopBank, the Turkish WordNet and UD 2.17 IMST), downloaded once into `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`) and checked against SHA-256 hashes. Taggers receive the **gold words**, so tokenisation errors do not enter the scores.

**Samples.** Instead of new annotation, the Kyrgyz sample is 50 random sentences of the human-annotated KyrgyzNER test set, mapped to PER/LOC/ORG with the mapping of Exercise 3; the Kazakh sample is 50 random KazNERD test sentences with the same KazNERD mapping. Both are fixed by seed 6. The LLM rows appear when an OpenAI-compatible server is configured through `LLM_MODELS` (comma-separated model names), `LLM_BASE_URL` and `LLM_API_KEY`. The toolkit tokenises the joined gold words itself; a sentence whose toolkit tokens do not match the gold words one-to-one is scored as all-O and counted as unaligned.

In [ ]:
import collections, hashlib, os, random, re, statistics
from pathlib import Path
import requests

# Pinned public releases (commit-addressed raw files) and their SHA-256 hashes.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_{}.txt"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/data/{}"
SHA256 = {
    KAZNERD.format("train"): "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD.format("valid"): "29d584d1a0d9aca6d58b39eb8a93e154791d303ecff56ef93f12b3c889239bf1",
    KAZNERD.format("test"): "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER.format("KyrgyzNER_TRAIN.jsonl.txt"): "719a74a18cd99d6709c1e0592d0337aafade65aa97fe762e62f3f72a6b343a26",
    KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"): "57a35a92139497819a2792d3e9e00a41b387b029b7851be7a64f4968befaddf5",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()

def fetch(url, sha256=None):
    """Download a pinned file once, check its hash and return the local path."""
    sha256 = sha256 or SHA256[url]
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha256:
        raise ValueError(f"{url} differs from the pinned release")
    return path

def read_conll(path, tag_col=-1, comment="#"):
    """Sentences as (words, tags) from a column file with blank-line sentence breaks."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith(comment + " "):
            cols = line.split("\t") if "\t" in line else line.split()
            words.append(cols[0]); tags.append(cols[tag_col].strip())
    return sents

def spans(tags):
    """Strict IOB2 spans {(start, end, type)}; an I- tag that does not continue a span starts none."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("I-") and typ == t[2:]:
            continue
        if typ is not None:
            out.add((start, i, typ))
        start, typ = (i, t[2:]) if t.startswith("B-") else (None, None)
    return out

def bioes_to_iob2(tags):
    return ["B-" + t[2:] if t[:2] in ("S-", "B-") else "I-" + t[2:] if t[:2] in ("E-", "I-") else "O" for t in tags]

def prf(gold, pred):
    """Micro precision, recall and F1 (%) over lists of span sets."""
    g = sum(len(x) for x in gold); p = sum(len(x) for x in pred)
    c = sum(len(x & y) for x, y in zip(gold, pred))
    pr = c / p if p else 0.0; rc = c / g if g else 0.0
    return {"gold": g, "pred": p, "correct": c, "P": round(100 * pr, 2), "R": round(100 * rc, 2),
            "F1": round(100 * (2 * pr * rc / (pr + rc) if pr + rc else 0.0), 2)}

def stanza_ner(lang="kk"):
    """Stanza NER over gold words; models from STANZA_RESOURCES_DIR, downloaded only if missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "ner").exists()
    return stanza.Pipeline(lang, processors="tokenize,ner", tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def tag_words(nlp, sents, batch=2000):
    """IOB2 predictions for gold-word sentences (spaces inside a token are joined with '_')."""
    out = []
    for i in range(0, len(sents), batch):
        doc = nlp([[w.replace(" ", "_") for w in words] for words, _ in sents[i:i + batch]])
        for s, (words, _) in zip(doc.sentences, sents[i:i + batch]):
            assert len(s.tokens) == len(words)
            out.append(bioes_to_iob2([t.ner for t in s.tokens]))
    return out

In [ ]:
from turkicnlp import Pipeline
# Two annotated samples of 50 sentences: KazNERD test (Kazakh) and KyrgyzNER test (Kyrgyz, human-annotated).
KAZ_MAP = {"PERSON": "PER", "GPE": "LOC", "LOCATION": "LOC", "ORGANISATION": "ORG"}
KIR_MAP = {"PERSON": "PER", "LOCATION": "LOC", "ORGANISATION": "ORG", "INSTITUTION": "ORG", "BUSINESS": "ORG"}
def to3(tags, table):
    return [f"{t[0]}-{table[t[2:]]}" if t != "O" and t[2:] in table else "O" for t in tags]

samples = {"kaz": (random.Random(6).sample(read_conll(fetch(KAZNERD.format("test"))), 50), KAZ_MAP),
           "kir": (random.Random(6).sample(read_conll(fetch(KYRGYZNER.format("KyrgyzNER_TEST.jsonl.gold.txt"))), 50), KIR_MAP)}

def report(label, gold, pred):
    r = prf(gold, pred)
    per = {t: prf([{x for x in g if x[2] == t} for g in gold], [{x for x in p if x[2] == t} for p in pred])["R"]
           for t in ("PER", "LOC", "ORG")}
    print(f"  {label:34} F1 {r['F1']:6.2f}  (P {r['P']}, R {r['R']}; gold {r['gold']}); recall by type {per}")

stanza_kk = stanza_ner("kk")
for lang, (sents, table) in samples.items():
    gold = [spans(to3(g, table)) for _, g in sents]
    print(f"{lang}: 50 sentences, {sum(map(len, gold))} PER/LOC/ORG mentions")
    report("Stanza KazNERD model", gold, [spans(to3(p, KAZ_MAP)) for p in tag_words(stanza_kk, sents)])
    for model in filter(None, os.environ.get("LLM_MODELS", "").split(",")):
        nlp = Pipeline(lang, processors=["tokenize", "ner"], tokenize_backend="rule", ner_backend="llm",
                       ner_labels=["PER", "LOC", "ORG"], llm_model=model.strip(),
                       llm_base_url=os.environ.get("LLM_BASE_URL"), llm_api_key=os.environ.get("LLM_API_KEY", "none"))
        pred, skipped = [], 0
        for ws, _ in sents:
            doc = nlp(" ".join(w.replace(" ", "_") for w in ws))
            tags = [w.ner or "O" for w in doc.words]
            if len(tags) != len(ws):                   # the toolkit split the text differently: no alignment
                skipped += 1; tags = ["O"] * len(ws)
            pred.append(spans(tags))
        report(f"LLM {model.strip()} ({skipped} unaligned)", gold, pred)
if not os.environ.get("LLM_MODELS"):
    print("No LLM configured: set LLM_MODELS (comma-separated), LLM_BASE_URL and LLM_API_KEY to add the LLM rows.")

## Worked solution

Output of `experiments/ch09_exercises/run_ch09_exercises.py` (no LLM server was configured in the environment used for the worked solution):

```
kaz: 50 sentences, 10 PER/LOC/ORG mentions
  Stanza KazNERD model               F1 100.00  (P 100.0, R 100.0; gold 10); recall by type {'PER': 100.0, 'LOC': 100.0, 'ORG': 100.0}
kir: 50 sentences, 29 PER/LOC/ORG mentions
  Stanza KazNERD model               F1  30.77  (P 34.78, R 27.59; gold 29); recall by type {'PER': 45.45, 'LOC': 27.27, 'ORG': 0.0}
No LLM configured: set LLM_MODELS (comma-separated), LLM_BASE_URL and LLM_API_KEY to add the LLM rows.
```

**The trained reference.** On the 50 Kazakh sentences the Stanza KazNERD model finds all 10 PER/LOC/ORG mentions (F1 100). Ten mentions are too few for a precise estimate, but the sample is in the model's own domain. On the 50 Kyrgyz sentences (29 mentions) the same model reaches F1 30.8: 45.5% recall for PER, 27.3% for LOC and none of the organisations, close to the full-test-set picture of Exercise 3 (F1 39.9).

**The LLM rows.** Run the cell with two models of different sizes. Compare them with these two references on the same mentions, and report per-type recall, to see whether errors concentrate in particular types or multiword names. Fifty sentences give only a coarse estimate. Report mention support and paired uncertainty; sample size alone does not determine whether a particular F1 difference is distinguishable from sampling variation.

## Your turn — extension

1. Configure a local server (vLLM or Ollama) and run two model sizes.
2. Annotate 50 Tatar news sentences yourself and repeat; compare your annotation decisions with the KyrgyzNER boundary policy.